In [1]:
MODEL = "qwen3.5:9b"
CONTEXT_SIZE = 32000

In [2]:
!sudo apt-get install zstd
!command -v ollama >/dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh
%pip install -q openai
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [3]:
import subprocess
import time
import os

env_val = os.getenv("OLLAMA_CONTEXT_LENGTH")
if env_val != str(CONTEXT_SIZE):
    print("Stopping any existing Ollama processes to apply new context length...")
    subprocess.run(["pkill", "-f", "ollama"], check=False)

os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXT_SIZE)
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

subprocess.Popen(["ollama", "serve"], env=os.environ.copy())

time.sleep(5)

subprocess.run(["ollama", "pull", MODEL], check=True)
print(f"Model '{MODEL}' pulled successfully.")

!ollama run $MODEL "Say Hey!"
!ollama ps

In [4]:
# from openai import OpenAI

# client = OpenAI(
#     base_url="http://127.0.0.1:11434/v1",
#     api_key="ollama"
# )

# response = client.chat.completions.create(
#     model=MODEL,
#     messages=[
#         {"role": "user", "content": "Say hey!"}
#     ]
# )

# print(response.choices[0].message.content)

In [5]:
!pkill -9 -f cloudflared || true

In [6]:
import subprocess
import re

proc = subprocess.Popen(
    [
        "./cloudflared-linux-amd64",
        "tunnel",
        "--url",
        "http://localhost:11434",
        "--http-host-header",
        "localhost:11434"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

url = None
while url is None:
    line = proc.stdout.readline()
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)

print(url)

In [7]:
# !pgrep -c cloudflared
# !curl http://127.0.0.1:11434/api/tags

In [ ]:
# import time
# import psutil
# import subprocess
# from collections import deque
# from IPython.display import clear_output
# import matplotlib.pyplot as plt
# import matplotlib as mpl

# # Dark dashboard theme
# mpl.rcParams.update({
#     "figure.facecolor": "#111827",
#     "axes.facecolor": "#1f2937",
#     "axes.edgecolor": "#6b7280",
#     "axes.labelcolor": "#f9fafb",
#     "axes.titlecolor": "#f9fafb",
#     "xtick.color": "#d1d5db",
#     "ytick.color": "#d1d5db",
#     "text.color": "#f9fafb",
#     "grid.color": "#d8d8d8",
#     "legend.facecolor": "#1f2937",
#     "legend.edgecolor": "#4b5563",
# })

# history = 300

# cpu_hist = deque(maxlen=history)
# ram_hist = deque(maxlen=history)
# gpu_hist = deque(maxlen=history)
# vram_hist = deque(maxlen=history)

# while True:
#     cpu_hist.append(psutil.cpu_percent())

#     ram = psutil.virtual_memory()
#     ram_hist.append(ram.percent)

#     try:
#         gpu_info = subprocess.check_output(
#             [
#                 "nvidia-smi",
#                 "--query-gpu=utilization.gpu,memory.used,memory.total",
#                 "--format=csv,noheader,nounits"
#             ]
#         ).decode().strip()

#         gpu_util, mem_used, mem_total = gpu_info.split(", ")

#         gpu_hist.append(float(gpu_util))
#         vram_hist.append(float(mem_used) / 1024)

#         vram_total = float(mem_total) / 1024

#     except Exception:
#         gpu_hist.append(0)
#         vram_hist.append(0)
#         vram_total = 0

#     clear_output(wait=True)

#     plt.figure(figsize=(14, 6))

#     plt.plot(cpu_hist, linewidth=2.5, label="CPU %")
#     plt.plot(ram_hist, linewidth=2.5, label="RAM %")
#     plt.plot(gpu_hist, linewidth=2.5, label="GPU %")

#     plt.ylim(0, 100)
#     plt.grid(True, alpha=0.3)

#     plt.title(
#         f"System Monitor | VRAM: {vram_hist[-1]:.1f} GB / {vram_total:.1f} GB",
#         pad=15
#     )

#     plt.xlabel("Last 5 Minutes")
#     plt.ylabel("Utilization (%)")
#     plt.legend(loc="upper left")

#     plt.tight_layout()
#     plt.show()

#     print("\n=== NVIDIA-SMI ===\n")

#     try:
#         smi = subprocess.check_output(
#             ["nvidia-smi"],
#             stderr=subprocess.STDOUT
#         ).decode()

#         print(smi)

#     except Exception as e:
#         print("Failed to run nvidia-smi:", e)

#     time.sleep(1)

In [ ]:
# import time
# import psutil
# from IPython.display import clear_output

# while True:
#     clear_output(wait=True)

#     print(f"CPU: {psutil.cpu_percent():.1f}%")
#     ram = psutil.virtual_memory()
#     print(f"RAM: {ram.percent:.1f}% ({ram.used/1024**3:.1f} GB / {ram.total/1024**3:.1f} GB)")

#     !nvidia-smi

#     time.sleep(1)